In [2]:
import sys

!{sys.executable} -m pip install \
transformers==4.37.2 \
accelerate==0.25.0 \
huggingface_hub==0.20.3 \
torch==2.2.2 \
--upgrade --force-reinstall

  Obtaining dependency information for transformers==4.37.2 from https://files.pythonhosted.org/packages/85/f6/c5065913119c41ecad148c34e3a861f719e16b89a522287213698da911fc/transformers-4.37.2-py3-none-any.whl.metadata
  Using cached transformers-4.37.2-py3-none-any.whl.metadata (129 kB)
  Obtaining dependency information for accelerate==0.25.0 from https://files.pythonhosted.org/packages/f7/fc/c55e5a2da345c9a24aa2e1e0f60eb2ca290b6a41be82da03a6d4baec4f99/accelerate-0.25.0-py3-none-any.whl.metadata
  Using cached accelerate-0.25.0-py3-none-any.whl.metadata (18 kB)
  Obtaining dependency information for huggingface_hub==0.20.3 from https://files.pythonhosted.org/packages/28/03/7d3c7153113ec59cfb31e3b8ee773f5f420a0dd7d26d40442542b96675c3/huggingface_hub-0.20.3-py3-none-any.whl.metadata
  Using cached huggingface_hub-0.20.3-py3-none-any.whl.metadata (12 kB)
  Obtaining dependency information for torch==2.2.2 from https://files.pythonhosted.org/packages/3f/14/e105b8ef6d324e789c1589e95cb0ab63

Using cached transformers-4.37.2-py3-none-any.whl (8.4 MB)
Using cached accelerate-0.25.0-py3-none-any.whl (265 kB)
Using cached huggingface_hub-0.20.3-py3-none-any.whl (330 kB)
Using cached torch-2.2.2-cp311-none-macosx_10_9_x86_64.whl (150.8 MB)
Using cached fsspec-2026.4.0-py3-none-any.whl (203 kB)
Using cached numpy-2.4.4-cp311-cp311-macosx_10_9_x86_64.whl (17.0 MB)
Using cached packaging-26.2-py3-none-any.whl (100 kB)
Using cached pyyaml-6.0.3-cp311-cp311-macosx_10_13_x86_64.whl (185 kB)
Using cached regex-2026.4.4-cp311-cp311-macosx_10_9_x86_64.whl (291 kB)
Using cached safetensors-0.7.0-cp38-abi3-macosx_10_12_x86_64.whl (467 kB)
Using cached tokenizers-0.15.2-cp311-cp311-macosx_10_12_x86_64.whl (2.6 MB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached filelock-3.29.0-py3-none-any.whl (39 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using c

In [2]:
import sys
!{sys.executable} -m pip install "numpy<2" --force-reinstall

  Obtaining dependency information for numpy<2 from https://files.pythonhosted.org/packages/11/57/baae43d14fe163fa0e4c47f307b6b2511ab8d7d30177c491960504252053/numpy-1.26.4-cp311-cp311-macosx_10_9_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 20.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tsinterpret 0.4.7 requires tqdm<4.66.0,>=4.61.2, but you have tqdm 4.67.3 which is incompatible.
pyfume 0.3.4 requires numpy==1.24.4, but you have numpy 1.26.4 which is incompatible.
pyfume 0.3.4 requires scipy==1.10.1, but you have scipy 1.14.0 which is incompatible.


In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
inputs = tokenizer("This movie is amazing!", return_tensors="pt")
outputs = model(**inputs)

print(outputs.logits)

tensor([[-0.0056,  0.0661]], grad_fn=<AddmmBackward0>)


In [12]:
import pandas as pd

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

train['label'] = train['label'].map({'positive': 1, 'negative': 0})

In [13]:
def tokenize_function(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=256)


from sklearn.model_selection import train_test_split

train = train.sample(5000, random_state=42).reset_index(drop=True)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train['review'],
    train['label'],
    test_size=0.2,
    random_state=42
)

train_encodings = tokenize_function(train_texts.tolist())
val_encodings = tokenize_function(val_texts.tolist())
test_encodings = tokenize_function(test['review'].tolist())

print("Train:", len(train_encodings['input_ids']), len(train_labels))
print("Val:", len(val_encodings['input_ids']), len(val_labels))

Train: 4000 4000
Val: 1000 1000


In [14]:
import torch

class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

In [15]:
train_dataset = IMDBDataset(train_encodings, train_labels.tolist())
val_dataset = IMDBDataset(val_encodings, val_labels.tolist())
test_dataset = IMDBDataset(test_encodings)

In [ ]:
#!{sys.executable} -m pip install accelerate

In [ ]:
# !{sys.executable} -m pip install \
# transformers==4.37.2 \
# accelerate==0.25.0 \
# huggingface_hub==0.20.3 \
# -U

In [ ]:
# import accelerate
# print(accelerate.__version__)


In [17]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm   # ⭐ 进度条

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

device = torch.device("cpu")
model.to(device)

model.train()

for epoch in range(3):
    print(f"\nEpoch {epoch}")
    
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_loader)
    print("Average Loss:", avg_loss)


Epoch 0


100%|█████████████████████████████████████████| 250/250 [28:59<00:00,  6.96s/it]


Average Loss: 0.3472796055674553

Epoch 1


100%|█████████████████████████████████████████| 250/250 [34:33<00:00,  8.30s/it]


Average Loss: 0.17110201609879733

Epoch 2


100%|███████████████████████████████████████| 250/250 [2:40:19<00:00, 38.48s/it]

Average Loss: 0.07750726115144789


In [18]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print("Validation Accuracy:", accuracy)

Validating: 100%|███████████████████████████████| 63/63 [02:12<00:00,  2.10s/it]

Validation Accuracy: 0.878


In [19]:
from tqdm import tqdm

model.eval()

preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        batch_preds = torch.argmax(outputs.logits, dim=1)

        preds.extend(batch_preds.cpu().numpy())

Predicting: 100%|█████████████████████████████| 313/313 [10:39<00:00,  2.04s/it]


In [20]:
pred_labels = ['positive' if p == 1 else 'negative' for p in preds]

submission = pd.DataFrame({
    'Id': test['Id'],
    'Label': pred_labels
})

submission.to_csv("prediction.csv", index=False)

In [ ]:
# print("test size:", len(test))
# print("preds size:", len(preds))
# print("test_loader batches:", len(test_loader))

In [ ]:
#print(len(preds), len(test))

In [ ]:
# 1. sample 2000 -> 5000

# 2. epoch 2 -> 3
# 3. max_length 128 -> 256
# 4. batch_size 8 -> 16

# 5. learning rate（2e-5 → 3e-5）
# 6. dropout
# 7. weight decay